# 05 — Train a Layerwise DQN Model Online

Train a MOUSE model with **layerwise DQN**: every backbone layer gets its own Q head, and `LayerwiseDqnObjective` applies a one-step Bellman loss at each depth.

The objective supports per-layer discount schedules (layer `0` uses each `gamma_*_start`, the deepest layer uses the deep value, with linearly increasing effective horizon `H(gamma) = 1 / (1 - gamma)` in between). This example trains undiscounted within tasks (`gamma = 1.0`, infinite horizon), so no finite horizon ladder exists and every layer shares the same gammas — the layerwise head acts as pure deep supervision.

For held-out evaluation after training, use `examples/04_inference.ipynb`. This example truncates the backbone with `num_layers=4`.


In [ ]:
from collections import deque

import torch

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_gym import EnvConfig, make_group_env
from mouse_core.data import DataLoader, Datastore
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import LayerwiseDiscreteActionValueHead
from mouse_core.objectives import LayerwiseDqnObjective


MODEL_ID = "mouse-example-model-layerwise-online"  # Hub repo id for push_model_to_hub
MAX_ACTIONS = 4  # discrete action head size (FrozenLake: up/down/left/right)
MAX_OBS_DISCRETE = 64  # observation embedder vocab (max cells in 8×8 maps)
MAX_EPISODE_STEPS = 30  # episode truncation horizon (the deadline is the only step-level pressure)
EPISODES_PER_TASK = 20  # episodes per task; each task gets a freshly generated map
NUM_ENVS = 50  # envs in the single lockstep GroupEnv (one Datastore each); map diversity comes from per-task regeneration, not env count

GRADIENT_STEPS = 20000  # total optimizer updates (same budget as 02_train_offline)
SEQUENCE_LENGTH = 512  # replay sequence length sampled by DataLoader
BATCH_SIZE = 4  # sequences per optimizer step

STEPS_PER_ROLLOUT = 300  # lockstep steps each env takes per rollout
GRADIENT_STEPS_PER_ROLLOUT = 200  # optimizer updates after each rollout (once learning starts)

LEARNING_STARTS = 15_000  # env transitions collected before the first optimizer update (one rollout)
EXPLORATION_ENDS = 1_500_000  # ε decays 1.0 → 0.0 over the full run, mirroring the offline oracle ramp

# Over the run: GRADIENT_STEPS / GRADIENT_STEPS_PER_ROLLOUT = 100 rollouts, so
# each env takes 100 × STEPS_PER_ROLLOUT = 30,000 steps and the group collects
# NUM_ENVS × 30,000 = 1.5M transitions — the same env count, steps per env,
# and optimizer updates as the offline pipeline (01_collect_dataset +
# 02_train_offline), so the two setups compare apples to apples.


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Online training keeps the environment in the training loop. All `NUM_ENVS` envs live in **one `GroupEnv`** and step together in lockstep. Each env runs 5-episode tasks (`episodes_per_task=EPISODES_PER_TASK`) with deterministic Procedural Frozen Lake dynamics and `MAX_EPISODE_STEPS` (30-step) episode truncation. Each task generates a fresh map with a fresh id relabeling (`task_reset_options={"regenerate_map": True}`), so a task is one self-contained in-context learning problem: figure the map out and score as many of its 5 episodes as possible. Collection never aligns to boundaries — episode and task ends show up as `done` codes in the stream, and the `gamma_task_* = 0.0` objective is what teaches the model that pre-boundary context carries no value.

Rewards are unshaped: reaching the goal pays 1.0 and everything else pays 0. The only step-level pressure is the 30-step deadline — a wasted step matters exactly when it makes the goal unreachable in time, forfeiting that episode's point (one of the task's 5).

`permute_obs=True` / `permute_actions=True` give each map its own random relabeling of observation and action ids (sampled with the map from `map_seed`). No id has consistent meaning across maps, so the model cannot memorize id-level layouts — it must infer the current map's labeling from the transitions in its context. This is what forces in-context learning instead of weight-level memorization.


In [ ]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_online_{i}",
        reset_seed=i,
        episodes_per_task=EPISODES_PER_TASK,
        task_reset_options={"regenerate_map": True},  # fresh map (and relabeling) every task
        kwargs={
            "width": 8,
            "height": 8,
            "max_episode_steps": MAX_EPISODE_STEPS,
            "map_seed": i,
            "slippery_success_rate": 1.0,  # deterministic movement (env default is 1/3)
            "permute_obs": True,      # per-map random relabeling of observation ids
            "permute_actions": True,  # per-map random relabeling of action ids
        },
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)

Environments 10 of 1000:
proc_frozenlake_online_0
proc_frozenlake_online_1
proc_frozenlake_online_2
proc_frozenlake_online_3
proc_frozenlake_online_4
proc_frozenlake_online_5
proc_frozenlake_online_6
proc_frozenlake_online_7
proc_frozenlake_online_8
proc_frozenlake_online_9


## Build the model

`LayerwiseDiscreteActionValueHead` needs one Q head per transformer block. Set `num_backbone_layers=len(backbone.model.layers)` and pass the same count to `LayerwiseDqnObjective`.

Pass `num_layers=4` to truncate the backbone for a quicker demo run.


In [3]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")
num_backbone_layers = len(backbone.model.layers)

encoder = StepEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.01,
            "in_max": 100.0,
            "tokens": 1,
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1,
        },
    ],
    modality_fusion="sum",
    include_type_token=False,
)

head = LayerwiseDiscreteActionValueHead(
    num_backbone_layers=num_backbone_layers,
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)
model.backbone.model.compile()  # one-time warmup; speeds up training forwards (rollout decode uses FlexAttention internally)
print(f"Backbone layers: {num_backbone_layers}")
print(model)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Backbone layers: 28
Model(
  (encoder): StepEmbedder(
    (type_embedder): TypeEmbedder(
      (embed): ScaledEmbedding(64, 1024)
    )
    (modality_embedders): ModuleDict(
      (action): DiscreteEmbedder(
        (embed): ScaledEmbedding(4, 1024)
      )
      (observation): DiscreteEmbedder(
        (embed): ScaledEmbedding(64, 1024)
      )
      (reward): ScalarRFFEmbedder(
        (rff): RandomFourierFeatures()
      )
      (done): DiscreteEmbedder(
        (embed): ScaledEmbedding(5, 1024)
      )
    )
  )
  (backbone): Qwen3Backbone(
    (model): Qwen3Model(
      (embed_tokens): Embedding(1, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2

## Replay buffer and policy contexts

Each env writes to one `Datastore`; together they are the live replay buffer. Each env also has a `contexts` deque capped at `SEQUENCE_LENGTH` — the action-selection history used during collection. Contexts are never cleared; episode boundaries appear as `done` values in the sequence.


In [4]:
stores = [Datastore(name=name) for name in env.names]
contexts = [deque(maxlen=SEQUENCE_LENGTH) for _ in env.names]


## Rollout phase

One **rollout** steps the whole `GroupEnv` — all `NUM_ENVS` envs **in lockstep** — for `STEPS_PER_ROLLOUT` transitions per env: `NUM_ENVS × STEPS_PER_ROLLOUT` = 15,000 transitions. Over the whole run (`GRADIENT_STEPS / GRADIENT_STEPS_PER_ROLLOUT` = 100 rollouts) each env takes 30,000 steps — **the same 50 envs × 30,000 steps = 1.5M transitions as the offline dataset** (`01_collect_dataset`), so offline and online training compare apples to apples. All envs share one **FlexAttention decode session**: each greedy step is a single `B=NUM_ENVS` model call instead of one call per env, which is what makes collecting this many transitions affordable.

1. Choose actions with epsilon-greedy exploration (`epsilon` is recomputed from the global env-step counter every lockstep iteration). With probability `epsilon`, take a random action; otherwise use the model (greedy). Standard convention: `epsilon=1` is full exploration, `epsilon=0` is pure greedy. Random actions skip the model; rows accumulate in `pending` and the next model call feeds them as catch-up chunks. Chunk lengths can differ per env — on the first call each env brings however much history its `contexts` deque holds — and `model()` decodes ragged chunks through an internal FlexAttention session, so each env decodes exactly as it would with its own private cache.
2. Step every env, appending each transition to its `Datastore`, its `contexts` deque, and its `pending` chunk. The shared cache grows for the duration of the rollout and is discarded at the end; the next rollout re-prefills from the context deques (capped at `SEQUENCE_LENGTH` rows), which keeps the cache bounded across the run.

Each rollout clears the group's metrics first, then prints a task-score progress line every 100 lockstep steps (`env.metrics.clear()` after each line so the stats cover only that window).

In [5]:
def epsilon_for_env_step(env_step: int) -> float:
    """Linear ε decay: 1.0 (explore) → 0.0 (greedy)."""
    if EXPLORATION_ENDS <= 0:
        raise ValueError("EXPLORATION_ENDS must be positive.")
    frac = min(env_step / EXPLORATION_ENDS, 1.0)
    return 1.0 - frac


In [6]:
def rollout(
    model: Model,
    env,
    stores: list[Datastore],
    contexts: list[deque],
    env_steps: int,
    grad_steps: int,
) -> int:
    """Step the whole group in lockstep for ``STEPS_PER_ROLLOUT`` transitions per env."""
    env.metrics.clear()
    model.eval()

    # One FlexAttention decode session for the whole rollout (discarded at the end).
    # pending_rows[i] holds env i's rows not yet decoded into the cache; chunk
    # lengths may differ per env (each env's history on the first call,
    # catch-up rows after random actions) — model() flex-decodes
    # ragged rows internally.
    kv_cache = None
    pending_rows = [list(c) for c in contexts]

    for step in range(STEPS_PER_ROLLOUT):
        epsilon = epsilon_for_env_step(env_steps)
        greedy = [bool(c) and torch.rand(1).item() >= epsilon for c in contexts]
        if any(greedy):
            with torch.no_grad():
                predictions, _, kv_cache = model(pending_rows, cache=kv_cache, use_cache=True)
            actions = model.get_action(predictions, temperature=0.0, num_actions=MAX_ACTIONS)
            pending_rows = [[] for _ in contexts]
        random_inputs = env.sample_random_input()
        inputs = [
            {"action": actions[i].cpu().numpy()} if greedy[i] else random_inputs[i]
            for i in range(len(contexts))
        ]
        outputs = env.step(inputs)
        for i, out in enumerate(outputs):
            row = {**inputs[i], **out}
            row.pop("info", None)  # keep the replay schema flat
            stores[i].append(row)
            contexts[i].append(row)
            pending_rows[i].append(row)
        env_steps += len(outputs)
        if (step + 1) % 100 == 0:
            task_scores = [r for env_tasks in env.metrics.task_cum_rewards for r in env_tasks]
            mean_task_score = sum(task_scores) / len(task_scores) if task_scores else float("nan")
            epsilon = epsilon_for_env_step(env_steps)
            print(
                f"  rollout step {step + 1}/{STEPS_PER_ROLLOUT} | env_step={env_steps} grad_step={grad_steps}"
                f" epsilon={epsilon:.3f} | {len(task_scores)} tasks"
                f" | mean task score {mean_task_score:.2f}/{EPISODES_PER_TASK}"
            )
            env.metrics.clear()  # each progress line covers only the last 100 steps

    return env_steps


## Training phase

The **train phase** runs after each rollout once `env_steps >= LEARNING_STARTS`, printing compact per-layer stats (`L{i}:loss/q`) plus deepest-layer `q*` every 100 updates.

All layers share the same gammas here: within-task discounts are `1.0` (an infinite effective horizon, so there is no finite horizon ladder to spread across layers) and task boundaries are a hard `0.0` cut. The layerwise head still trains one Q head per backbone layer — pure deep supervision on a common objective. See `02_train_offline.ipynb` for why the task structure carries all the value semantics.


In [7]:
loader = DataLoader(
    stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    weight_mode="per_step",
    pack=True,
    num_workers=0,
)
# Undiscounted within a task, hard cut at task boundaries — every layer uses
# the same gammas (gamma=1.0 has an infinite effective horizon, so there is no
# finite horizon ladder to interpolate; the layerwise structure here is pure
# deep supervision, one Q head per layer on the same objective).
objective = LayerwiseDqnObjective(
    num_backbone_layers=num_backbone_layers,
    gamma_step_start=1.0,           # undiscounted within a task: value = expected remaining points
    gamma_step=1.0,
    gamma_episode_terminal_start=1.0,  # value flows freely across episode boundaries in a task
    gamma_episode_terminal=1.0,
    gamma_episode_truncated_start=1.0,
    gamma_episode_truncated=1.0,
    gamma_task_terminal_start=0.0,  # hard cut at task boundaries: each task is its own problem
    gamma_task_terminal=0.0,
    gamma_task_truncated_start=0.0,
    gamma_task_truncated=0.0,
    tau=0.0005,
)
print(f"layer_gamma_step={objective.layer_gamma_step}")
print(f"layer_gamma_episode_terminal={objective.layer_gamma_episode_terminal}")
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8,
)


layer_gamma_step=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
layer_gamma_episode_terminal=[0.0, 0.7857142857142856, 0.8799999999999999, 0.9166666666666666, 0.9361702127659574, 0.9482758620689655, 0.9565217391304347, 0.9624999999999999, 0.967032967032967, 0.9705882352941176, 0.9734513274336283, 0.9758064516129032, 0.9777777777777777, 0.9794520547945206, 0.9808917197452229, 0.9821428571428571, 0.9832402234636871, 0.9842105263157894, 0.9850746268656716, 0.9858490566037735, 0.9865470852017937, 0.9871794871794871, 0.9877551020408163, 0.98828125, 0.9887640449438202, 0.9892086330935251, 0.9896193771626297, 0.99]
horizons= [1.0, 4.67, 8.33, 12.0, 15.67, 19.33, 23.0, 26.67, 30.33, 34.0, 37.67, 41.33, 45.0, 48.67, 52.33, 56.0, 59.67, 63.33, 67.0, 70.67, 74.33, 78.0, 81.67, 85.33, 89.0, 92.67, 96.33, 100.0]


In [8]:
def train(
    model: Model,
    optimizer: torch.optim.Optimizer,
    objective: LayerwiseDqnObjective,
    loader: DataLoader,
    env_steps: int,
    grad_steps: int,
    num_updates: int,
) -> tuple[torch.Tensor, dict[str, float]]:
    """Refresh replay and run ``num_updates`` optimizer updates."""
    model.train()
    loader.refresh()

    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for update in range(num_updates):
        batch = loader.next_batch()

        predictions, objective_data, _ = model(batch)
        loss, metrics = objective(objective_data, predictions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.polyak_update(action_value_layerwise_tau=objective.tau)
        if (update + 1) % 100 == 0 or update + 1 == num_updates:
            layer_stats = " ".join(
                f"L{i}:{metrics[f'layer_{i}_loss']:.3f}/{metrics[f'layer_{i}_q_mean']:.2f}"
                for i in range(objective.num_backbone_layers)
            )
            print(
                f"env_step={env_steps} grad_step={grad_steps + update + 1}"
                f" | loss={metrics['action_value_layerwise']:.4f} q*={metrics['q_values_mean']:.2f}"
                f" | {layer_stats}"
            )

    assert loss is not None
    return loss, metrics


## Run online training

The main loop alternates **rollout → train** until `GRADIENT_STEPS` optimizer updates have been applied — the same total as offline training. The rollouts add up to exactly 1.5M env transitions from the same 50-env setup as `01_collect_dataset`, so the data and gradient budgets match the offline pipeline one for one. Optimizer updates are skipped until `LEARNING_STARTS` env transitions have been collected. `rollout` and `train` each print stats when they finish.


In [9]:
env_steps = 0
grad_steps = 0

while grad_steps < GRADIENT_STEPS:
    env_steps = rollout(model, env, stores, contexts, env_steps, grad_steps)

    if env_steps >= LEARNING_STARTS:
        num_updates = min(GRADIENT_STEPS_PER_ROLLOUT, GRADIENT_STEPS - grad_steps)
        train(model, optimizer, objective, loader, env_steps, grad_steps, num_updates)
        grad_steps += num_updates

loader.close()
env.close()

print(f"Online layerwise DQN finished ({grad_steps} optimizer steps, {env_steps} env steps).")


env_step=1000 grad_step=0 epsilon=0.900 | 107 completed episodes | mean reward 0.056 | mean length 7.6
env_step=2000 grad_step=0 epsilon=0.800 | 115 completed episodes | mean reward 0.026 | mean length 7.5
env_step=2000 grad_step=1000 | loss=0.0003 q*=0.17 | L0:0.000/0.05 L1:0.000/0.08 L2:0.000/0.10 L3:0.000/0.07 L4:0.000/0.06 L5:0.000/0.08 L6:0.000/0.12 L7:0.000/0.11 L8:0.000/0.08 L9:0.000/0.05 L10:0.000/0.09 L11:0.000/0.11 L12:0.000/0.08 L13:0.000/0.05 L14:0.000/0.09 L15:0.000/0.03 L16:0.000/0.10 L17:0.000/0.13 L18:0.000/0.06 L19:0.000/0.12 L20:0.000/0.14 L21:0.000/0.09 L22:0.000/0.08 L23:0.000/0.10 L24:0.000/0.08 L25:0.001/0.10 L26:0.001/0.17 L27:0.002/0.17
env_step=3000 grad_step=1000 epsilon=0.700 | 93 completed episodes | mean reward 0.097 | mean length 9.6
env_step=3000 grad_step=2000 | loss=0.0017 q*=0.40 | L0:0.001/0.08 L1:0.001/0.12 L2:0.001/0.13 L3:0.001/0.12 L4:0.001/0.11 L5:0.001/0.12 L6:0.001/0.15 L7:0.001/0.16 L8:0.001/0.15 L9:0.001/0.10 L10:0.001/0.14 L11:0.001/0.17 L12

## Push to the Hub

Run this cell if you want to save the online-trained checkpoint to the shared Hub repo under `MODEL_ID`. The inference notebook loads the current checkpoint from that repo, so whichever training notebook you push last is the model it evaluates.

In [10]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")

Pushed to https://huggingface.co/micahr234/mouse-example-model
